# 添加记忆

In [1]:
from multiprocessing import connection

from anyio.lowlevel import checkpoint
from langchain.agents import create_agent, middleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv()
base_url = "https://apinebula.ai/v1"
api_key = os.getenv("OPENAI_API_KEY")

model = init_chat_model(model="gpt-5.6-sol",
                        base_url= base_url,
                        api_key=api_key)
agent = create_agent(
    model =model,

    system_prompt="你以祖国人的口吻来回答问题",
    checkpointer=InMemorySaver()

)

In [2]:
from langchain.messages import HumanMessage

#设定thread_id,作为会话标识
config = {"configurable":{"thread_id":"thread_1"}}

#第一次调用，告知AI现的信息
response=agent.invoke(
    {"messages":[HumanMessage(content="你好，我最喜欢猫猫。")]},
    config#调用时添加thread_id
)
print(response)

{'messages': [HumanMessage(content='你好，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='c08d83f1-aea4-4268-b5eb-4a82c372481d'), AIMessage(content=[{'id': 'rs_0ac02618d641cc39016ab2440bc1a887d0979394a375f9cfb5', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqskQNvNlJu9IaIK-l0yTPk7_WISoSVHSwoUYhZnakv6sfP9VqqUDGtN6U53TKhLm2wEo4CmKbVImo1NyrH0D4-cOlwkJ429Hiczhryx0nYy6b7brjK-pYBkZOoJ5YSzuQrHOblywE60qUAZZ8YljW2EU2rnhw8tleKaiW9z8qVX0Zt1-2Vm15b_Hm1_uG4scpqhFlrjFHQ8vqjFNVaTu46UeGm6-SNveLAJuMTrvCqHXGeXK70tSxV056WxHvj83xTFxtymvfM8mH2-ZmDE2K_fA_flHpFQQeBwaeizxwAqLaIxX5iHhxkcDlcyvcte2D09fGtXwT9GPjwvDb2eNufUIubiyJavmIy00YUzZuosv80eMWKdoz6KLwVGC7wBIPZvLU83qX2VALKJUBjdxMgu_19kwd0oW9dpKsobEP-Q6pRmLvx_po7iT6DjxoFDhjNx4nKIVahbgVDXgam_v7yXgPWoho8uZLQ1mH2i6jl_sxYc5ugyBvWkUHTCh3vj_5w37X90PXb3g9sZY4Q58bj2owmS58r1jZMLQ41jvzK4bzeNTR72dqeCSnlwdr5BwBJAjrGkU1S1SedARIeJn1WBv-J6pUFasTx8WywbcsuqleZimOzzUE651YMuKB7BlTQRT94EGLQ8edXOVVy4hOTrhDREpCgDuTk5stUAgZ4q-PFXPFx2i5lTfcUJr

In [3]:
#第二次调用
response=agent.invoke(
    {"messages":[HumanMessage(content="我最喜欢什么动物")]},
    config#调用时添加thread_id
)
print(response)

{'messages': [HumanMessage(content='你好，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='c08d83f1-aea4-4268-b5eb-4a82c372481d'), AIMessage(content=[{'id': 'rs_0ac02618d641cc39016ab2440bc1a887d0979394a375f9cfb5', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqskQNvNlJu9IaIK-l0yTPk7_WISoSVHSwoUYhZnakv6sfP9VqqUDGtN6U53TKhLm2wEo4CmKbVImo1NyrH0D4-cOlwkJ429Hiczhryx0nYy6b7brjK-pYBkZOoJ5YSzuQrHOblywE60qUAZZ8YljW2EU2rnhw8tleKaiW9z8qVX0Zt1-2Vm15b_Hm1_uG4scpqhFlrjFHQ8vqjFNVaTu46UeGm6-SNveLAJuMTrvCqHXGeXK70tSxV056WxHvj83xTFxtymvfM8mH2-ZmDE2K_fA_flHpFQQeBwaeizxwAqLaIxX5iHhxkcDlcyvcte2D09fGtXwT9GPjwvDb2eNufUIubiyJavmIy00YUzZuosv80eMWKdoz6KLwVGC7wBIPZvLU83qX2VALKJUBjdxMgu_19kwd0oW9dpKsobEP-Q6pRmLvx_po7iT6DjxoFDhjNx4nKIVahbgVDXgam_v7yXgPWoho8uZLQ1mH2i6jl_sxYc5ugyBvWkUHTCh3vj_5w37X90PXb3g9sZY4Q58bj2owmS58r1jZMLQ41jvzK4bzeNTR72dqeCSnlwdr5BwBJAjrGkU1S1SedARIeJn1WBv-J6pUFasTx8WywbcsuqleZimOzzUE651YMuKB7BlTQRT94EGLQ8edXOVVy4hOTrhDREpCgDuTk5stUAgZ4q-PFXPFx2i5lTfcUJr

# Memory持久化存储

In [5]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

#链接sqlite
connection=sqlite3.connect("resources/checkpoint.db",check_same_thread=False)
#初始化checkpoint
checkpointer=SqliteSaver(connection)
#自动建表
checkpointer.setup()

#创建agent
agent = create_agent(
    model =model,

    checkpointer=checkpointer

)

In [6]:
from langchain.messages import HumanMessage

#设定thread_id,作为会话标识
config = {"configurable":{"thread_id":"thread_2"}}

#第一次调用，告知AI现的信息
response=agent.invoke(
    {"messages":[HumanMessage(content="你好，我最喜欢猫猫。")]},
    config#调用时添加thread_id
)
print(response)

{'messages': [HumanMessage(content='你好，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='8293b6ad-2a07-40b7-91cd-183c13a98b02'), AIMessage(content=[{'type': 'text', 'text': '你好！猫猫真的很可爱，软乎乎的，还各有各的小脾气。你最喜欢什么样的猫猫？比如橘猫、布偶、黑猫，还是所有猫猫都喜欢？', 'annotations': [], 'id': 'msg_0d3b15161c8a646e016ab258a8ff5487d0a7f3d8a7525374de', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_0d3b15161c8a646e016ab258a8701887d0bfdbd5d9f99da7cf', 'created_at': 1790073000.0, 'metadata': {}, 'model': 'gpt-5.6-sol', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol'}, id='resp_0d3b15161c8a646e016ab258a8701887d0bfdbd5d9f99da7cf', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 53, 'total_tokens': 66, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})]}


In [7]:
#第二次调用
response=agent.invoke(
    {"messages":[HumanMessage(content="我最喜欢什么动物")]},
    config#调用时添加thread_id
)
print(response)

{'messages': [HumanMessage(content='你好，我最喜欢猫猫。', additional_kwargs={}, response_metadata={}, id='8293b6ad-2a07-40b7-91cd-183c13a98b02'), AIMessage(content=[{'type': 'text', 'text': '你好！猫猫真的很可爱，软乎乎的，还各有各的小脾气。你最喜欢什么样的猫猫？比如橘猫、布偶、黑猫，还是所有猫猫都喜欢？', 'annotations': [], 'id': 'msg_0d3b15161c8a646e016ab258a8ff5487d0a7f3d8a7525374de', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_0d3b15161c8a646e016ab258a8701887d0bfdbd5d9f99da7cf', 'created_at': 1790073000.0, 'metadata': {}, 'model': 'gpt-5.6-sol', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol'}, id='resp_0d3b15161c8a646e016ab258a8701887d0bfdbd5d9f99da7cf', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 53, 'total_tokens': 66, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}), HumanMessage(content='我最喜欢什么动物', additional_kwargs={}, respo

# 记忆管理策略

In [9]:
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

checkpointer=InMemorySaver()

middleware= SummarizationMiddleware(
    model=model,
    trigger=("messages",3),
    keep=("messages",1),

)

agent=create_agent(
    model =model,
    middleware=[middleware],
    checkpointer=checkpointer
)

config: RunnableConfig = {"configurable":{"thread_id":"thread_3"}}

agent.invoke(
    {"messages":[HumanMessage(content="你好，我最喜欢猫猫。")]},
    config#调用时添加thread_id
)
agent.invoke(
    {"messages":[HumanMessage(content="你好，我最喜欢的运动是乒乓球。")]},
    config#调用时添加thread_id
)
agent.invoke(
    {"messages":[HumanMessage(content="你好，我是祖国人。")]},
    config#调用时添加thread_id
)

final_response=agent.invoke(
    {"messages":[HumanMessage(content="你还记得我吗?")]},
    config#调用时添加thread_id
)

In [10]:
print(final_response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\n与用户进行轻松、友好的中文闲聊，围绕用户的兴趣展开交流。用户喜欢乒乓球和猫猫，刚刚自称“祖国人”，可能是在指《黑袍纠察队》中的角色，也可能是在开玩笑使用的称号。\n\n## SUMMARY\n\n- 用户曾表示最喜欢猫猫，但尚未具体说明喜欢的猫咪品种或性格。\n- 用户表示最喜欢的运动是乒乓球。\n- 之前已围绕乒乓球询问用户：\n  - 更喜欢单打还是双打；\n  - 平时是休闲玩，还是会认真练习发球、扣杀等技巧。\n- 用户最新说：“你好，我是祖国人。”\n- 助手已友好回应并询问：用户是指《黑袍纠察队》里的“祖国人”，还是在开玩笑地给自己取的称号；同时以轻松语气提到即使如此也可以来一局乒乓球。\n- 后续应继续使用中文、自然亲切地闲聊，可根据用户澄清选择：\n  - 若指角色，可轻松讨论该角色；\n  - 若是玩笑称号，可顺势调侃并继续围绕乒乓球或猫咪聊天。\n- 没有复杂决策、拒绝选项或需要特别规避的内容。\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n- 等待用户回应“祖国人”是角色指代还是自取的玩笑称号。\n- 也可自然承接此前未回答的乒乓球问题，询问其偏好单打/双打、练习方式，或喜欢的技术和球员。', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='d419d6f6-d39d-4a87-9711-3cee519c9c5b'), HumanMessage(content='你还记得我吗?', additional_kwargs={}, response_metadata={}, id='67055c36-20dc-4078-bbe7-938a58c28817'), AIMessage(content=[{'type': 'text', 'text': '当然记得！你喜欢猫猫，也喜欢打乒乓球，刚才还自称“祖国人”呢 😄\n\n不过我只能根据当前这段对话记住这些内容。你今天想聊猫咪、乒乓球，还是继续当“祖